# 02 - Skill Analysis

Analyse skill demand: top skills, categories, skills by role/location, and skill combinations.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

from src.data_loader import load_raw_data, detect_and_rename_columns
from src.data_cleaning import clean_data
from src.skill_extractor import extract_skills
from src.feature_engineering import engineer_features
from src import analytics

In [ ]:
# Load and process
df = load_raw_data()
df = detect_and_rename_columns(df)
cleaned = clean_data(df)
cleaned = extract_skills(cleaned)
processed = engineer_features(cleaned)
print(f"Processed shape: {processed.shape}")

## Top Skills

In [ ]:
top = analytics.get_top_skills(processed, n=20)
top.plot(kind='barh', x='skill', y='count', figsize=(10, 8), title='Top 20 skills')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Skill Category Distribution

In [ ]:
from src.skill_extractor import SKILL_CATEGORIES
categories = {}
for skills in processed['extracted_skills']:
    for s in skills:
        cat = SKILL_CATEGORIES.get(s, 'Other')
        categories[cat] = categories.get(cat, 0) + 1

pd.Series(categories).sort_values(ascending=False).plot(kind='bar', figsize=(10, 5), title='Skill categories')
plt.tight_layout()
plt.show()

## Skills by Role

In [ ]:
roles = processed['standardized_job_title'].unique()
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, role in zip(axes.flatten(), roles[:4]):
    role_df = processed[processed['standardized_job_title'] == role]
    skills = analytics.get_top_skills(role_df, n=10)
    if len(skills):
        skills.plot(kind='barh', x='skill', y='count', ax=ax, title=role)
        ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Skills by Location

In [ ]:
banglore = processed[processed['city'] == 'Bangalore']
skills = analytics.get_top_skills(banglore, n=15)
skills.plot(kind='barh', x='skill', y='count', figsize=(10, 6), title='Top skills in Bangalore')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Skill Combinations

In [ ]:
combos = analytics.get_skill_combinations(processed, n=10)
combos['combination'] = combos['skill_1'] + ' + ' + combos['skill_2']
combos.plot(kind='barh', x='combination', y='count', figsize=(10, 6), title='Top skill combinations')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()